# btp-teacher-targets
Training-only target generation after persistent/cold equivalence and exact budget approval.

Implementation workflow; no completed GPU experiment is implied. Historical notebooks are preserved.

## Source and runtime
Attach the private research source bundle. Internet is needed to provision the isolated runtime; running this cell downloads dependencies. CPU smoke precedes GPU use. No long experiment runs automatically.

In [ ]:
import hashlib
import json
import os
import shutil
import subprocess
import sys
import time
import zipfile
from pathlib import Path

ROOT = Path("/kaggle/working/BTP")
OUTPUT = Path("/kaggle/working") / ("btp-research-" + str(time.time_ns()))
OUTPUT.mkdir()
# Upload artifacts/btp-kaggle.zip as btp-research-source.bundle in a private dataset.
bundles = list(Path("/kaggle/input").rglob("btp-research-source.bundle"))
if not ROOT.exists():
    assert len(bundles) == 1, (
        "Attach one research source bundle; see docs/RESEARCH_WORKFLOWS.md"
    )
    with zipfile.ZipFile(bundles[0]) as archive:
        for info in archive.infolist():
            destination = (ROOT.parent / info.filename).resolve()
            assert destination.is_relative_to(ROOT), "Unsafe source member"
        archive.extractall(ROOT.parent)
snapshot = json.loads((ROOT / "source-manifest.json").read_text())
for name, expected in snapshot["files"].items():
    path = (ROOT / name).resolve()
    assert path.is_relative_to(ROOT) and path.is_file(), name
    assert hashlib.sha256(path.read_bytes()).hexdigest() == expected, name
print("Verified source files:", len(snapshot["files"]))

In [ ]:
uv = shutil.which("uv")
if uv is None:
    subprocess.run([sys.executable, "-m", "pip", "install", "uv"], check=True)
UV = [uv] if uv else [sys.executable, "-m", "uv"]
PYTHON = Path("/tmp/btp-research-env/bin/python")
if not PYTHON.exists():
    subprocess.run(
        UV + ["venv", "--python", "3.10", str(PYTHON.parent.parent)], check=True
    )
subprocess.run(
    UV
    + [
        "pip",
        "install",
        "--python",
        str(PYTHON),
        "torch==2.6.0",
        "torchvision==0.21.0",
        "--index-url",
        "https://download.pytorch.org/whl/cu118",
    ],
    check=True,
)
subprocess.run(
    UV
    + [
        "pip",
        "install",
        "--python",
        str(PYTHON),
        "-r",
        str(ROOT / "configs/kaggle-requirements.txt"),
    ],
    check=True,
)
RUN = UV + ["run", "--no-project", "--python", str(PYTHON), "python"]
ENVIRONMENT = dict(os.environ, PYTHONPATH=str(ROOT / "src"), CUDA_VISIBLE_DEVICES="0")


def run(arguments):
    subprocess.run(RUN + arguments, cwd=ROOT, env=ENVIRONMENT, check=True)


with (OUTPUT / "runtime-packages.txt").open("w") as handle:
    subprocess.run(
        UV + ["pip", "freeze", "--python", str(PYTHON)], stdout=handle, check=True
    )
(OUTPUT / "source-manifest.json").write_text(json.dumps(snapshot, indent=2))

Validate cold M0 versus resident output and A-B-A state reset first. This stage stays disabled until the exact training pair list/budget is approved. Clean targets are never passed to the teacher.

In [ ]:
RUN_TARGETS = False
TRAIN_MANIFEST = None
WEIGHTS = None
APPROVED_PAIR_IDS = []
if RUN_TARGETS:
    assert TRAIN_MANIFEST is not None and WEIGHTS is not None and APPROVED_PAIR_IDS
    settings = dict(
        manifest=str(TRAIN_MANIFEST),
        weights=str(WEIGHTS),
        pairs=APPROVED_PAIR_IDS,
        max_attempts=8,
        max_seconds=1800,
    )
    (OUTPUT / "target-settings.json").write_text(json.dumps(settings, indent=2))
    script = """import json, sys
from pathlib import Path
from btp.resident import ResidentConfig, ResidentTeacher
from btp.teacher_targets import generate_targets, TargetBudget
root, output = Path(sys.argv[1]), Path(sys.argv[2])
s = json.loads((output/'target-settings.json').read_text()); p = Path(s['manifest'])
evaluation = json.loads((root/'configs/evaluation/nh_haze_manifest.json').read_text())
with ResidentTeacher(ResidentConfig(root/'upstream/Learning-Hazing-to-Dehazing/configs/inference/stage2_accsamp.yaml'), Path(s['weights'])) as teacher:
    report = generate_targets(document=json.loads(p.read_text()), root=p.parent,
        output=output/'targets', teacher=teacher, approved_pair_ids=s['pairs'],
        budget=TargetBudget(s['max_attempts'], s['max_seconds']), evaluation_scenes=evaluation['scenes'], seed=42)
print(json.dumps(report, indent=2))
"""
    run(["-c", script, str(ROOT), str(OUTPUT)])
else:
    print(
        "Disabled: resident equivalence, audited corpus and exact budget approval required."
    )

In [ ]:
archive = shutil.make_archive(str(OUTPUT), "zip", OUTPUT)
print("Download evidence:", archive)